In [2]:
import pandas as pd
import numpy as np

def ip_to_int(ip):
    """Convert IP address to integer, handling NaN and invalid values"""
    if pd.isna(ip):
        return np.nan
    try:
        ip_str = str(ip)
        parts = ip_str.split('.')
        if len(parts) != 4:
            return np.nan
        return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])
    except:
        return np.nan

# Load data
df_fraud = pd.read_csv('../data/raw/Fraud_Data.csv')
df_ip = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print(f"Fraud data: {df_fraud.shape}")
print(f"IP data: {df_ip.shape}")

# Check ip_address column for non-string values
print(f"IP address column type: {df_fraud['ip_address'].dtype}")
print(f"Sample IP values: {df_fraud['ip_address'].head(10).tolist()}")

# Convert IP to integer (handle errors)
df_fraud['ip_int'] = df_fraud['ip_address'].apply(ip_to_int)
df_ip['lower_int'] = df_ip['lower_bound_ip_address'].apply(ip_to_int)
df_ip['upper_int'] = df_ip['upper_bound_ip_address'].apply(ip_to_int)

# Drop rows with NaN IPs
df_fraud = df_fraud.dropna(subset=['ip_int'])
df_ip = df_ip.dropna(subset=['lower_int', 'upper_int'])

print(f"Fraud data after IP conversion: {df_fraud.shape}")
print(f"IP data after conversion: {df_ip.shape}")

# Merge using merge_asof (requires sorting)
df_ip_sorted = df_ip.sort_values('lower_int')
df_fraud_sorted = df_fraud.sort_values('ip_int')

df_merged = pd.merge_asof(df_fraud_sorted, df_ip_sorted, left_on='ip_int', right_on='lower_int', direction='backward')

# Filter where ip_int <= upper_int
df_merged = df_merged[df_merged['ip_int'] <= df_merged['upper_int']]

print(f"Merged shape: {df_merged.shape}")

# Check fraud by country
fraud_by_country = df_merged[df_merged['class'] == 1]['country'].value_counts().head(10)
print("Top 10 countries with fraud:")
print(fraud_by_country)

# Save enriched data
df_merged.to_csv('../data/processed/fraud_data_with_country.csv', index=False)
print("Saved enriched fraud data")

Fraud data: (151112, 11)
IP data: (138846, 3)
IP address column type: float64
Sample IP values: [732758368.79972, 350311387.865908, 2621473820.11095, 3840542443.91396, 415583117.452712, 2809315199.92675, 3987484328.51882, 1692458727.64945, 3719094257.18731, 341674739.579911]
Fraud data after IP conversion: (0, 12)
IP data after conversion: (0, 5)
Merged shape: (0, 17)
Top 10 countries with fraud:
Series([], Name: count, dtype: int64)
Saved enriched fraud data
